In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf

In [ ]:
from keras.layers import Input,Dense,Flatten
from keras.models import Model
from keras.optimizers import Adam
from keras.preprocessing import image
import numpy as np
import glob
from keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore',category=FutureWarning)
from datetime import datetime
from keras.callbacks import ModelCheckpoint
from keras import applications
from keras.applications import MobileNetV2


In [ ]:
# Define the image size
IMAGE_SIZE = [224, 224, 3]

# Load the model
mobilenetv2 = MobileNetV2(include_top=False, input_shape=IMAGE_SIZE, weights='imagenet')

# Visualize the model summary
#mobilenetv2.summary()


9406464/9406464 [==============================] - 1s 0us/step


In [ ]:
for layer in mobilenetv2.layers:
    layer.trainable = False

In [ ]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
# Add custom head to the base model

x = mobilenetv2.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

In [ ]:
model = Model(inputs=mobilenetv2.input, outputs=predictions)

In [ ]:

adam=Adam()

model.compile(loss='binary_crossentropy',
              optimizer=adam,
              metrics=['accuracy'])

In [4]:
train_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Train/BinaryTrain'
test_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Test/Binarytest'

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_datagen=ImageDataGenerator(preprocessing_function=preprocess_input)

# Train data
train_set = train_datagen.flow_from_directory(train_path,
                                              target_size=(224, 224),
                                              batch_size=64,
                                              class_mode='binary')

# Test data
test_set = test_datagen.flow_from_directory(test_path,
                                            target_size=(224, 224),
                                            batch_size=64,
                                            class_mode = 'binary')



Found 2409 images belonging to 2 classes.
Found 604 images belonging to 2 classes.


In [ ]:
output_layer = model.layers[-1]  # Assuming the output layer is the last layer in the model
num_classes = output_layer.output_shape[-1]  # Number of units in the output layer

print("Number of classes in the output layer:", num_classes)


Number of classes in the output layer: 1


Class weights

In [ ]:
import os
from sklearn.utils import class_weight
import numpy as np

# Path to your dataset
dataset_path = train_path

# Function to count images in each class directory
def count_images_in_classes(dataset_path):
    labels = []
    class_names = os.listdir(dataset_path)
    for class_index, class_name in enumerate(class_names):
        class_dir = os.path.join(dataset_path, class_name)
        if os.path.isdir(class_dir):
            num_images = len([img_name for img_name in os.listdir(class_dir) if os.path.isfile(os.path.join(class_dir, img_name))])
            labels.extend([class_index] * num_images)
    return np.array(labels), class_names

# Count images and get labels
y_train, class_names = count_images_in_classes(dataset_path)

# Compute class weights
class_weights = class_weight.compute_class_weight('balanced',
                                                  classes=np.unique(y_train),
                                                  y=y_train)

class_weights_dict = dict(zip(np.unique(y_train), class_weights))

# Print computed class weights
print("Computed class weights:")
print(class_weights_dict)


Computed class weights:
{0: 1.9028436018957346, 1: 0.6782094594594594}


In [ ]:
print(class_names)

['WaterFilledPotholes', 'PotHoles']


In [ ]:
batch_size=64

# Calculate steps_per_epoch

steps_per_epoch = train_set.samples // batch_size
if train_set.samples % batch_size != 0:
    steps_per_epoch += 1
print(steps_per_epoch)
# Calculate validation_steps
val_set=test_set
validation_steps = val_set.samples // batch_size
if val_set.samples % batch_size != 0:
    validation_steps += 1
print(validation_steps)

38
10


In [ ]:
def get_subfolder_names_alt(directory_path):
    subfolders = [item for item in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, item))]
    return subfolders

# Example usage:
folder_path = train_path
subfolder_names_alt = get_subfolder_names_alt(folder_path)
print(subfolder_names_alt)


['WaterFilledPotholes', 'PotHoles']


In [ ]:
from tensorflow.keras.utils import to_categorical

class_mapping = {'WaterFilledPotholes':0, 'PotHoles':1}
# Assuming 'y_train' contains your original labels
y_train_encoded = [class_mapping[label] for label in y_train]


num_classes = len(class_mapping)
y_train_encoded = to_categorical(y_train_encoded, num_classes=num_classes)


In [ ]:
# Define the file name for the model checkpoint
checkpoint_filepath = '/content/drive/MyDrive/Models/mobilenetv2_binary.keras'

# Define the ModelCheckpoint callback
checkpoint = ModelCheckpoint(filepath=checkpoint_filepath, verbose=1, save_best_only=True)

# Combine all callbacks
callbacks = [checkpoint]

# Start timing
start = datetime.now()

# Train the model
model_history = model.fit(train_set,
                          validation_data=test_set,
                          epochs=5,
                          class_weight=class_weights_dict,
                          steps_per_epoch = 38,
                          validation_steps = 10,
                          callbacks=callbacks)
# Calculate duration5
duration = datetime.now() - start

print('Total elapsed time:', duration)

In [1]:
from keras.models import load_model
model = load_model("/content/drive/MyDrive/Models/mobilenetv2_binary.keras")


In [2]:
import os

# Replace 'path_to_your_directory' with the actual path to your directory
directory_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Test/Binarytest'

# Get all folder names in the directory
class_labels= [f for f in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, f))]

print(class_labels)

['WaterFilledPotholes', 'PotHoles']


In [ ]:
from keras.preprocessing import image
from keras.applications.imagenet_utils import decode_predictions
import numpy as np

your_directory = test_path
y_true=[]
y_pred=[]
for root, dirs, files in os.walk(your_directory):
    for file in files:
      img_path=os.path.join(root, file)

      remove=your_directory+'/'
      orginal_label=root.replace(remove,"")
      y_true.append(orginal_label)

      # Load and preprocess the image
      img = image.load_img(img_path, target_size=(224, 224))  # Replace target size with your model's expected input size
      img_array = image.img_to_array(img)
      img_array = np.expand_dims(img_array, axis=0)

      # Predict the class probabilities
      probabilities = model.predict(img_array)

      # Get the index of the class with the highest probability
      predicted_index = np.argmax(probabilities)

      # Get the name of the class
      predicted_class = class_labels[predicted_index]
      y_pred.append(predicted_class)



print(y_true)
print(y_pred)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn import metrics

# Assuming you have the true labels and predicted labels
true_labels = y_true# Replace with the true labels from your test set
predicted_labels =y_pred # Replace with the predicted labels from your model

# Create the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='g', xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true,y_pred))